# Protein Language Model B-cell Epitope Classification

End-to-end Kaggle workflow using the project's existing Python modules. Choose either `bcell` or `sars`, then set `cfg.MODEL_TYPE` to `protbert`, `esm2`, or `prott5`; the matching model and tokenizer are selected automatically. Choose one `cfg.CLASSIFIER_TYPE` (`pooled_mlp`, `cnn`, `attention_pooling`, or `bigru`) and one `cfg.FEATURE_FUSION` for each run. The configuration cell exposes fusion, classifier, LoRA, regularization, and training hyperparameters. Alternative heads require `cfg.USE_SMOTE=False`; the separate embedding-SMOTE path uses its own MLP. ProtT5 uses space-separated residues and mean-pools residue embeddings. B-cell uses stratified 5-fold protein-level cross-validation; SARS uses stratified peptide-level 5-fold cross-validation because it has one protein. SARS folds share a source protein and do not measure performance on unseen proteins. Imputation/scaling are fit on each fold's training rows and SMOTE is applied only to training rows.

Clone this repository into `/kaggle/working` and attach the CSV dataset as a Kaggle input. The modules are imported from the `TF` package.

**Suggested fusion experiment order:** start with `sequence_only`, then compare `weighted_concat`, `gated_residual`, `film`, `se`, `gmu`, `cross_attention`, `late_fusion`, and `low_rank_bilinear`. Keep model, classifier head, folds, seed, loss, optimizer, threshold, and preprocessing fixed while changing only fusion. For architecture comparisons, use `cfg.USE_SMOTE=False`. Compare out-of-fold PR-AUC/ROC-AUC and threshold-based metrics; do not select using a held-out test set.

In [ ]:
!git clone https://github.com/azhar416/B-Cell-Epitopes.git

In [ ]:
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version
import importlib.util
import subprocess
import sys
from IPython.display import display

PROJECT_DIR = Path("/kaggle/working/B-Cell-Epitopes")
if not (PROJECT_DIR / "TF" / "CFG.py").is_file():
    raise FileNotFoundError(f"Project Python files were not found in {PROJECT_DIR}")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

if importlib.util.find_spec("imblearn") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "imbalanced-learn>=0.12,<1"])

from TF.CFG import CFG
from TF.pre_dataset import load_protein_tokenizer, resolve_max_length
from TF.dataset import prepare_dataset
from TF.train import train_validation_split, set_seed
import torch

In [ ]:
cfg = CFG()
cfg.set_debug(debug=False)

# DATASET
cfg.DATASET_NAME = "bcell"  # choose "bcell" or "sars"

# MODEL
cfg.MODEL_TYPE = "esm2"  # choose "protbert", "esm2", or "prott5"
cfg.CLASSIFIER_TYPE = "cnn"  # choose one: pooled_mlp, cnn, attention_pooling, bigru
cfg.USE_DIPEPTIDE_FEATURES = False  # Add 400-dimensional DPC from peptide_seq
if cfg.MODEL_TYPE == "prott5" and importlib.util.find_spec("sentencepiece") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "sentencepiece>=0.1.99"])
cfg.CNN_CHANNELS = 128  # Try 32, 64, 128; "legacy" uses a model-width-based value
cfg.CNN_DROPOUT = 0.3  # CNN head only
cfg.CLASSIFIER_HIDDEN_SIZE = None  # Try 128, 256, 512; None uses half encoder width
cfg.CLASSIFIER_DROPOUT = 0.5  # Try 0.3, 0.5, 0.7

# PROTEIN CONTEXT 
cfg.USE_PROTEIN_FEATURES = True  # Set True to enable flanking protein context
cfg.FLANKING_SIZE = 10             # Residues on N- and C-terminal flanks
cfg.MAX_PEPTIDE_LENGTH = 20

# CROSS-VALIDATION / TRAINING
cfg.SEED = 42  # Same seed and protein-level folds for comparisons
cfg.BATCH_SIZE = 32
cfg.EPOCHS = 100
cfg.EARLY_STOPPING_PATIENCE = 10
cfg.EARLY_STOPPING_MIN_DELTA = 0.001
cfg.SCHEDULER_PATIENCE = 3
cfg.SCHEDULER_FACTOR = 0.5
cfg.SCHEDULER_MIN_LR = 1e-7
cfg.VALIDATION_SIZE = 0.2  # Used by holdout helpers; CV remains 5 folds
cfg.PREDICTION_THRESHOLD = 0.5
cfg.NUM_WORKERS = 0

# OVERSAMPLING
cfg.USE_SMOTE = False  # Pre-split SMOTE in frozen encoder-embedding + descriptor space.
cfg.SMOTE_K_NEIGHBORS = 5

# FEATURE FUSION
cfg.FEATURE_FUSION = "cross_attention"  # sequence_only, weighted_concat, cross_attention, se, gmu, film, late_fusion, low_rank_bilinear, gated_residual
cfg.CROSS_ATTENTION_HEADS = 8  # cross_attention only; must divide encoder hidden size
cfg.CROSS_ATTENTION_DROPOUT = 0.1
cfg.FEATURE_FUSION_DROPOUT = 0.1  # weighted_concat, film, gated_residual descriptor branches
cfg.LATE_FUSION_DROPOUT = 0.5  # late_fusion descriptor branch
cfg.LOW_RANK_BILINEAR_DIM = 32  # low_rank_bilinear only; try 16, 32, 64

# LOSS FUNCTION
cfg.LOSS_FUNCTION = "binary_focal"  # "binary_focal" or "bce"
if cfg.LOSS_FUNCTION == 'binary_focal':
    cfg.FOCAL_ALPHA = 0.75
    cfg.FOCAL_GAMMA = 2.0
cfg.LABEL_SMOOTHING = 0.05  # Try 0, 0.05, 0.1

# REGULARIZER
cfg.MSD_ENABLED = True  # Independent regularizer; enable only for an MSD experiment.
if cfg.MSD_ENABLED:
    cfg.MSD_SAMPLES = 5
    cfg.MSD_DROPOUT = 0.3
else:
    cfg.MSD_SAMPLES = 1  # Must be >=2 when MSD_ENABLED=True

# ENCODER / LORA / OPTIMIZATION
cfg.USE_LORA = True  # Set False for frozen encoder-head-only baseline
cfg.FREEZE_ENCODER = True  # Keep True with LoRA; set False for full fine-tuning
cfg.LORA_RANK = 4  # Try 2, 4, 8
cfg.LORA_ALPHA = 8
cfg.LORA_DROPOUT = 0.2
cfg.LORA_LEARNING_RATE = 5e-6
cfg.ENCODER_LEARNING_RATE = 2e-5  # Used for full fine-tuning when LoRA is off
cfg.HEAD_LEARNING_RATE = 1e-4
cfg.WEIGHT_DECAY = 0.1
cfg.GRADIENT_CLIP_NORM = 1.0
cfg.USE_MULTI_GPU = True
# Encoder-specific target names: ProtT5 uses q/v; ProtBERT/ESM2 use query/value.
cfg.PROTT5_LORA_TARGET_MODULES = ("q", "v")
cfg.LORA_TARGET_MODULES = ("query", "value")
if cfg.USE_LORA and not cfg.USE_SMOTE:
    if importlib.util.find_spec("peft") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "peft"])
    try:
        torchao_version = version("torchao")
    except PackageNotFoundError:
        torchao_version = None
    if torchao_version is not None:
        from packaging.version import Version

        if Version(torchao_version) < Version("0.16.0"):
            subprocess.check_call(
                [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"]
            )
            print(f"Removed incompatible optional torchao {torchao_version}; PEFT LoRA does not need it.")
cfg._DATASET_PATH = "/kaggle/input/datasets/daffamazhar/b-cell-epitope"  # change if your Kaggle input path differs
cfg.CHECKPOINT_PATH = str(
    Path("/kaggle/working/TF-artifacts")
    / cfg.MODEL_TYPE
    / ("flanked" if cfg.USE_PROTEIN_FEATURES else "no_flanking")
    / ("smote_embedding" if cfg.USE_SMOTE else cfg.FEATURE_FUSION)
    / ("pooled_mlp" if cfg.USE_SMOTE else cfg.CLASSIFIER_TYPE)
    / ("dpc" if cfg.USE_DIPEPTIDE_FEATURES else "no_dpc")
    / ("no_msd" if cfg.USE_SMOTE else (f"msd_{cfg.MSD_SAMPLES}" if cfg.MSD_ENABLED else "no_msd"))
    / "classifier.pt"
)

In [ ]:
set_seed(cfg.SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dataset: {cfg.DATASET_NAME} | SMOTE: {cfg.USE_SMOTE} | Fusion: {cfg.FEATURE_FUSION} | Classifier: {cfg.CLASSIFIER_TYPE} | Device: {device}")
print(f"Checkpoint and plots: {cfg.CHECKPOINT_PATH}")

## Prepare the dataset

Peptide sequences and labels are validated, contradictory labels for identical peptides are removed, exact duplicate peptides are collapsed, and peptides longer than 20 residues are excluded. B-cell validation is selected by complete `parent_protein_id` groups; the deterministic group split approximates the requested row ratio and label prevalence. SARS validation is stratified by peptide label because all current rows share one protein.

In [ ]:
tokenizer = load_protein_tokenizer(
    cfg.MODEL_TYPE,
    cfg.MODEL_NAME,
    tokenizer_name=cfg.PROTT5_TOKENIZER_NAME,
)
cfg.MAX_LENGTH = resolve_max_length(tokenizer, cfg.EFFECTIVE_PEPTIDE_LENGTH)
flanking_info = f"flanking={cfg.FLANKING_SIZE}" if cfg.USE_PROTEIN_FEATURES else "no flanking"
print(f"Tokenized max length: {cfg.MAX_LENGTH} ({cfg.EFFECTIVE_PEPTIDE_LENGTH} residues [{flanking_info}] + {tokenizer.num_special_tokens_to_add(pair=False)} special tokens)")
dataset = prepare_dataset(
    dataset_name=cfg.DATASET_NAME,
    tokenizer=tokenizer,
    data_dir=cfg._DATASET_PATH,
    max_length=cfg.MAX_LENGTH,
    model_type=cfg.MODEL_TYPE,
    max_peptide_length=cfg.MAX_PEPTIDE_LENGTH,
    use_protein_features=cfg.USE_PROTEIN_FEATURES,
    flanking_size=cfg.FLANKING_SIZE,
    use_dipeptide_features=cfg.USE_DIPEPTIDE_FEATURES,
)

print(f"Total examples: {len(dataset):,}")
print("Labels [0, 1]:", torch.bincount(dataset.labels.long(), minlength=2).tolist())


## Train the selected experiment

Set `cfg.USE_SMOTE=True` to use the separate embedding-SMOTE experiment: frozen ProtBert/ESM2 first-token or ProtT5 mean-pooled residue embeddings and descriptors are computed, then imputation/scaling are fit on each fold's training rows and SMOTE is applied only to those rows before an MLP is trained. This mode does not use LoRA, selectable fusion layers, MSD, or alternate classifier heads. Set it to `False` to use the standard trainable sequence pipeline, where one classifier head and one feature-fusion mode can be selected. B-cell folds hold out complete proteins; SARS folds share the same source protein.

In [ ]:
split_results = train_validation_split(
    dataset=dataset,
    cfg=cfg,
    device=device,
)
from IPython.display import Image
print("Split strategy:", split_results["split_strategy"])
print("Validation interpretation:", split_results["validation_metrics_interpretation"])
def format_metrics(metrics):
    count_metrics = {"tn", "fp", "fn", "tp"}
    return {
        metric: (
            value
            if metric in count_metrics or not isinstance(value, (int, float))
            else f"{value:.4f}"
        )
        for metric, value in metrics.items()
    }

def format_mean_std(mean_metrics, std_metrics):
    return {
        metric: f"{mean:.4f} ± {std_metrics[metric]:.4f}"
        for metric, mean in mean_metrics.items()
    }

print("Mean ± std across folds — training:")
display(format_mean_std(split_results["mean_training_metrics"], split_results["std_training_metrics"]))
print("Mean ± std across folds — evaluation:")
display(format_mean_std(split_results["mean_validation_metrics"], split_results["std_validation_metrics"]))
print("Pooled out-of-fold evaluation metrics (combined predictions, not mean ± std):")
display(format_metrics(split_results["validation_metrics"]))
for fold in split_results["fold_metrics"]:
    print(f"Fold {fold['fold']} — training metrics:")
    display(format_metrics(fold["training_metrics"]))
    print(f"Fold {fold['fold']} — evaluation metrics:")
    display(format_metrics(fold["validation_metrics"]))
    display(Image(filename=fold["training_history_plot"]))

## Validation results

Each fold uses early stopping on validation ROC-AUC and reduces learning rates when validation ROC-AUC plateaus. Pooled out-of-fold metrics summarize all held-out folds; they are cross-validation results, not an independent external test.

In [ ]:
print("Out-of-fold predictions:", split_results["out_of_fold_predictions"])
print("Metrics report:", split_results["metrics_path"])